### INGEST FROM BRONZE LAYER

In [0]:
df = spark.table('workspace.bronze_schema.sales_info_delta_1')
df.display()

###RENAME COLUMN NAMES

In [0]:
Sales_dict = {'sls_ord_num': 'Sales_order_number', 'sls_prd_key': 'Sales_product_key', 'sls_cust_id': 'Sales_customer_id', 'sls_order_dt': 'Sales_order_date', 'sls_ship_dt': 'Sales_ship_date', 'sls_due_dt': 'Sales_due_date', 'sls_sales': 'Sales_sales', 'sls_quantity': 'Sales_quantity', 'sls_price': 'Total_Sales_price'}

for key, value in Sales_dict.items():
    df = df.withColumnRenamed(key, value)
df.display()


In [0]:
df.printSchema()

### CHECK FOR NULLS

In [0]:
from pyspark.sql.functions import col, sum

null_counts_df = df.select(
    *[
        sum(col(c).isNull().cast('int')).alias(c)
        for c in df.columns
    ]
)

null_counts_df.display()


### CHANGE INTEGER FORMAT FOR DATETIME, THEN CHANGE FORMAT

In [0]:
from pyspark.sql.functions import col, to_date, try_to_date

df = df.withColumn(
    'Sales_order_date',
    try_to_date(col('Sales_order_date').cast('string'), 'yyyyMMdd')
)

df.display()



### CONVERT MULTIPLE COLUMNS INTO DATE

In [0]:


date_cols = [
    'Sales_ship_date',
    'Sales_due_date'
]

for c in date_cols:
    df = df.withColumn(
        c,
        try_to_date(col(c).cast('string'), 'yyyyMMdd')
    )
df.display()

In [0]:
df.printSchema()

### FILL NA FOR SALES_PRICE

In [0]:
df.fillna({'Sales_price': 0})\
    .fillna({'Sales_sales': 0})\
    
df.display()

In [0]:

null_counts_df = df.select(
    *[
        sum(col(c).isNull().cast('int')).alias(c)
        for c in df.columns
    ]
)

null_counts_df.display()

In [0]:
df.write.mode('overwrite').saveAsTable('workspace.silver_schema.sales_info')